# Competition Name - Modeling

## Overview
- Competition: [Competition Name](https://www.kaggle.com/competitions/xxx)
- Model: LightGBM baseline

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, accuracy_score
from pathlib import Path

from src.utils import seed_everything, reduce_mem_usage

seed_everything(42)

pd.set_option('display.max_columns', 100)

In [ ]:
# Paths
INPUT_DIR = Path('../input')
OUTPUT_DIR = Path('../output')

# Load data
train = pd.read_csv(INPUT_DIR / 'train.csv')
test = pd.read_csv(INPUT_DIR / 'test.csv')

## Feature Engineering

In [ ]:
TARGET_COL = 'target'
ID_COL = 'id'

FEATURE_COLS = [col for col in train.columns if col not in [TARGET_COL, ID_COL]]

## Cross Validation

In [ ]:
N_FOLDS = 5

lgb_params = {
    'objective': 'regression',  # or 'binary', 'multiclass'
    'metric': 'rmse',           # or 'auc', 'multi_logloss'
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

In [ ]:
X = train[FEATURE_COLS]
y = train[TARGET_COL]
X_test = test[FEATURE_COLS]

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
feature_importance = pd.DataFrame()

kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(kfold.split(X, y)):
    print(f'\n--- Fold {fold + 1} ---')
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
    
    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=10000,
        valid_sets=[train_data, valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=100)
        ]
    )
    
    oof_preds[valid_idx] = model.predict(X_valid)
    test_preds += model.predict(X_test) / N_FOLDS
    
    # Feature importance
    fold_importance = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance': model.feature_importance(importance_type='gain'),
        'fold': fold + 1
    })
    feature_importance = pd.concat([feature_importance, fold_importance])

# Overall CV score
cv_score = np.sqrt(mean_squared_error(y, oof_preds))
print(f'\nOverall CV RMSE: {cv_score:.5f}')

## Feature Importance

In [ ]:
import matplotlib.pyplot as plt

mean_importance = feature_importance.groupby('feature')['importance'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 12))
plt.barh(range(min(30, len(mean_importance))), mean_importance.head(30).values[::-1])
plt.yticks(range(min(30, len(mean_importance))), mean_importance.head(30).index[::-1])
plt.xlabel('Importance')
plt.title('Top 30 Feature Importance')
plt.tight_layout()
plt.show()

## Submission

In [ ]:
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET_COL: test_preds
})

submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'Submission saved: {submission.shape}')
submission.head()